# Balanced PCA-plane activation rewrite pipeline

This notebook is a configuration-driven wrapper around the reusable Python scripts. Running all cells fits the aggregated three-component PCA and balanced separating plane, saves its artifacts, then rewrites every source activation row using its own plane-projected PCA score.

In [5]:
# CONFIGURATION — edit this cell only.
from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'scripts' / 'fit_balanced_pca_plane.py').is_file():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')


ROOT = find_repo_root()

# Input and output paths
INPUT_ACTS_DIR = ROOT / '.acts_filter_2'
REWRITTEN_ACTS_DIR = ROOT / '.acts_filter_3'
ARTIFACT_DIR = ROOT / 'results' / 'balanced_pca_plane'
METADATA_CACHE_DIR = ROOT / 'data' / 'activation_explorer_cache'

# Output artifact filenames (all are written beneath ARTIFACT_DIR)
PCA_FILENAME = 'pca_3_components_filter_3.joblib'
CLASSIFIER_FILENAME = 'balanced_linear_svm_filter_3.joblib'
POINTS_FILENAME = 'original_and_projected_pc_points_filter_3.csv'
PLOT_FILENAME = 'pca_scores_and_plane_filter_3.html'

# Optional fitted three-component PCA .joblib. Set to None to fit PCA on this run.
# The supplied model is still saved beneath ARTIFACT_DIR using PCA_FILENAME.
PREFITTED_PCA_PATH = r"..\results\balanced_pca_plane\pre_fit_filter_3_activation_pca_layer_out-21.joblib"

# Folder-to-class grouping. Values must be the binary labels 0 and 1.
# Dictionary order controls folder discovery and plot legend order.
FOLDER_TO_CLASS = {
    'plain': 1,
    'plain_long': 0,
    'indirect': 0,
    'new_conv': 1,
}

# Raw rows matching ANY listed field/value are ignored during aggregation, PCA fitting,
# and classifier fitting. They are still transformed and written to REWRITTEN_ACTS_DIR.
IGNORE_WHEN_FITTING = {
    'base_unit': ['millennia', 'seconds'],
}

# Model and execution settings
PCA_BATCH_SIZE = 4096
SVM_C = 1.0
RANDOM_STATE = 42
MAX_PLOT_POINTS_PER_FOLDER = 5_000
OVERWRITE_REWRITTEN_BATCHES = True

## 1. Fit PCA and the balanced linear plane

Activations are averaged by `(task, source_folder, time_horizon_months)` before PCA. Classifier sample weights give every configured folder equal total representation.

In [6]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.fit_balanced_pca_plane import fit_balanced_pca_plane
from scripts.rewrite_activations_with_projected_pca import rewrite_activation_batches


artifact_paths = fit_balanced_pca_plane(
    INPUT_ACTS_DIR,
    ARTIFACT_DIR,
    cache_dir=METADATA_CACHE_DIR,
    folders=tuple(FOLDER_TO_CLASS),
    class_by_folder=FOLDER_TO_CLASS,
    ignore_filters=IGNORE_WHEN_FITTING,
    prefitted_pca_path=PREFITTED_PCA_PATH,
    pca_batch_size=PCA_BATCH_SIZE,
    svm_c=SVM_C,
    random_state=RANDOM_STATE,
    max_plot_points_per_folder=MAX_PLOT_POINTS_PER_FOLDER,
    pca_filename=PCA_FILENAME,
    classifier_filename=CLASSIFIER_FILENAME,
    points_filename=POINTS_FILENAME,
    plot_filename=PLOT_FILENAME,
)
artifact_paths

Metadata cache: C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\data\activation_explorer_cache
PCA: loaded from C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\balanced_pca_plane\pre_fit_filter_3_activation_pca_layer_out-21.joblib
Ignored 10,268 rows; aggregated 201,975 fit rows into 4,784 points.
Classifier accuracy: 0.475334
Folder-weighted classifier accuracy: 0.475334
Folder-weighted balanced accuracy: 0.475334
              precision    recall  f1-score   support

     class_1       0.48      0.52      0.50 2391.9999999999995
     class_2       0.47      0.43      0.45 2391.9999999999995

    accuracy                           0.48 4783.999999999999
   macro avg       0.48      0.48      0.47 4783.999999999999
weighted avg       0.48      0.48      0.47 4783.999999999999



{'pca': WindowsPath('C:/Users/91967/Desktop/AISC/temporal-manifolds-last-position/results/balanced_pca_plane/pca_3_components_filter_3.joblib'),
 'classifier': WindowsPath('C:/Users/91967/Desktop/AISC/temporal-manifolds-last-position/results/balanced_pca_plane/balanced_linear_svm_filter_3.joblib'),
 'points': WindowsPath('C:/Users/91967/Desktop/AISC/temporal-manifolds-last-position/results/balanced_pca_plane/original_and_projected_pc_points_filter_3.csv'),
 'plot': WindowsPath('C:/Users/91967/Desktop/AISC/temporal-manifolds-last-position/results/balanced_pca_plane/pca_scores_and_plane_filter_3.html')}

## 2. Rewrite every activation row

Each row is independently transformed with the fitted PCA, orthogonally projected onto the fitted plane, and moved in activation space by the corresponding reconstruction difference. The output payload is identical to its source payload except for the replaced activation tensor.

In [7]:
rewrite_summary = rewrite_activation_batches(
    INPUT_ACTS_DIR,
    REWRITTEN_ACTS_DIR,
    pca_path=artifact_paths['pca'],
    classifier_path=artifact_paths['classifier'],
    folders=tuple(FOLDER_TO_CLASS),
    overwrite=OVERWRITE_REWRITTEN_BATCHES,
)
rewrite_summary

{'output_dir': WindowsPath('C:/Users/91967/Desktop/AISC/temporal-manifolds-last-position/.acts_filter_3'),
 'files': 1661,
 'verified_files': 1661,
 'rows': 212243,
 'maximum_plane_error': 5.551115123125783e-17}

## Outputs

In [8]:
print('Model artifacts:')
for name, path in artifact_paths.items():
    print(f'  {name}: {path}')
print(f"Rewritten batches: {rewrite_summary['files']:,}")
print(f"Verified batches on disk: {rewrite_summary['verified_files']:,}")
print(f"Rewritten rows: {rewrite_summary['rows']:,}")
print(f"Activation output root: {rewrite_summary['output_dir']}")
if not Path(rewrite_summary['output_dir']).is_dir():
    raise FileNotFoundError('The reported activation output directory is not present on disk.')

Model artifacts:
  pca: C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\balanced_pca_plane\pca_3_components_filter_3.joblib
  classifier: C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\balanced_pca_plane\balanced_linear_svm_filter_3.joblib
  points: C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\balanced_pca_plane\original_and_projected_pc_points_filter_3.csv
  plot: C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\balanced_pca_plane\pca_scores_and_plane_filter_3.html
Rewritten batches: 1,661
Verified batches on disk: 1,661
Rewritten rows: 212,243
Activation output root: C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\.acts_filter_3
